# Creating a Medial Axis

__Author(s):__ Cinar Turhan and Masa Prodanovic

__Last Update:__ Jan. 2026

Copyright © 2026 Digital Porous Media Team. All rights reserved.

---

This notebook demonstrates how to extract the medial axis.

In [1]:
# Install packages if you havent:
# !pip install numpy scikit-image pyvista pyvista[jupyter]

In [2]:
# Import the required packages
%pip install --quiet --upgrade "dpm_tools>=1.0.2"
import sys
import site

# reload path just in case
sys.path.extend(site.getsitepackages())

import dpm_tools
import cc3d
import numpy as np
import matplotlib.pyplot as plt
import skimage
import pyvista as pv
pv.set_jupyter_backend('html')
pv.start_xvfb()

import warnings
warnings.filterwarnings('ignore')

from copy import deepcopy
import glob
import scipy
import os
from pathlib import Path
import sys

sys.path.append('../')

from dpm_tools.io import read_image, Image
from dpm_tools.visualization import plot_isosurface, extract_competent_subset, plot_medial_axis
import dpm_tools

import warnings
warnings.filterwarnings('ignore')

from copy import deepcopy
import glob
import scipy
from pathlib import Path

from dpm_tools.io import read_image, Image
from dpm_tools.visualization import plot_isosurface, extract_competent_subset, plot_medial_axis

import os
import numpy as np
import skimage 
import pyvista as pv
pv.set_jupyter_backend('html')
# pv.start_xvfb()
from pathlib import Path
import sys
sys.path.append('../')
from utils.dataloader import load_sample

download_path = Path("/home/jovyan/work/ls6")

Note: you may need to restart the kernel to use updated packages.


In [3]:
def plot_sample(sample, subset=True, subset_range=(0, 128)):
    
    plotter_obj = pv.Plotter(lighting='three lights')
    # Set background colors
    plotter_obj.set_background(color='w')
    # Set font colors and sizes
    pv.global_theme.font.color = 'black'
    pv.global_theme.font.size = 18
    pv.global_theme.font.label_size = 14
    
    pv.set_jupyter_backend('html')
    
    if subset:
        # Check if diagonal mode (simple tuple) or exhaustive mode (nested tuples)
        if isinstance(subset_range[0], tuple):
            # Exhaustive mode: ((x_min, x_max), (y_min, y_max), (z_min, z_max))
            x_range, y_range, z_range = subset_range
            sample = sample[x_range[0]:x_range[1], 
                          y_range[0]:y_range[1], 
                          z_range[0]:z_range[1]]
        else:
            # Diagonal mode: (min, max)
            mini = subset_range[0]
            maxi = subset_range[1]
            sample = sample[mini:maxi, mini:maxi, mini:maxi]
    
    # sample = np.pad(sample, ((1, 1), (1, 1), (1, 1)), mode='constant', constant_values=1)
    sample = Image(scalar=sample)
    
    plotter_obj = plot_isosurface(sample, plotter_obj, show_isosurface=[0.5], 
                    mesh_kwargs={"opacity":1, 
                                "color":(200 / 255, 181 / 255, 152 / 255), 
                                "diffuse": 0.75, 
                                "ambient": 0.15})
    
    
    plotter_obj.show(jupyter_backend='html')

In [4]:
beadpack, castlegate = [load_sample(sample, cache_dir=download_path) for sample in ['beadpack', 'castlegate']]

Loading cached sample 'beadpack' from ../data/beadpack.tif
Loading cached sample 'castlegate' from ../data/castlegate.tif


In [5]:
# Select a subset from the data for easier visualization
subset1,_ = extract_competent_subset(beadpack,cube_size=150, pore_class=1, class_to_optimize=1)
subset2,_ = extract_competent_subset(castlegate,cube_size=100, pore_class=1, class_to_optimize=1)

Finding valid subsets:   0%|          | 0/100 [00:00<?]

Original Porosity (class 1): 37.88 %
Subset Porosity: 40.62 %
Optimized for class 1 connectivity
Competent Subset: [118:268,118:268, 118:268]


Finding valid subsets:   0%|          | 0/100 [00:00<?]

Original Porosity (class 1): 20.61 %
Subset Porosity: 20.92 %
Optimized for class 1 connectivity
Competent Subset: [53:153,53:153, 53:153]


### 3D Visualization
#### 1. Bead Pack

In [6]:
# Close the boundaries of the image:
beadpack_subset = beadpack[subset1[0]:subset1[1], subset1[0]:subset1[1], subset1[0]:subset1[1]]
beadpack_pad = np.pad(beadpack_subset, ((1, 1), (1, 1), (1, 1)), mode='constant', constant_values=1)

# Visualize
plot_sample(beadpack_pad, subset=False)

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

In [7]:
# Close the boundaries of the image:
castlegate_subset = castlegate[subset2[0]:subset2[1], subset2[0]:subset2[1], subset2[0]:subset2[1]]
castlegate_pad = np.pad(castlegate_subset, ((1, 1), (1, 1), (1, 1)), mode='constant', constant_values=1)

# Visualize
plot_sample(castlegate_pad, subset=False)

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

### Medial Axis Extraction
#### 1. Bead Pack

In [8]:
# Get medial axis
beadpack_medial_axis = skimage.morphology.skeletonize(beadpack_subset)

# Plot
mesh_kwargs = {'line_width':2, 'style':'wireframe', 'color': 'r'}
plot_sample(beadpack_medial_axis, mesh_kwargs)

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html>\n  <head>\n    <meta http-equiv=&quot;Content-…

#### Make it Fancier:

In [9]:
beadpack_medial_axis = plot_medial_axis(beadpack_subset, pore_class=1, interactive=True)
beadpack_medial_axis.show()

Widget(value='<iframe src="http://localhost:42123/index.html?ui=P_0x7f900a4981d0_3&reconnect=auto" class="pyvi…

### 2. Sandstone

In [10]:
castlegate_medial_axis = plot_medial_axis(castlegate_subset, pore_class=1, interactive=True)
castlegate_medial_axis.show()

Widget(value='<iframe src="http://localhost:42123/index.html?ui=P_0x7f900a17f8d0_4&reconnect=auto" class="pyvi…